# AI DLC — Fix Agent, live

One AI agent takes a plain-English task, finds the problem in a real repository,
changes the code, runs that repository's own tests, and opens a pull request **for you
to approve**. It cannot merge, and nothing reaches GitHub until you say so.

About 5 minutes. Run the cells in order.

---

### Before you start, you need three things

| | |
|---|---|
| **Access to the repo** | Dipesh adds you to `dipesharma29/agentic-ai-sdlc` on GitHub |
| **A GitHub token** | [Create one](https://github.com/settings/personal-access-tokens) with *Contents: read & write* and *Pull requests: read & write* on that repo |
| **Access to the GCP project** | Dipesh grants your Google account **Vertex AI User** on `ai-sdlc-515307` |

Then add your token to this notebook's secrets: the **key icon** in the left sidebar,
**+ Add new secret**, name it `GITHUB_TOKEN`, paste the value, and switch on
*Notebook access*.

> Your token stays in your own Colab secrets. It is never printed, never written to a
> file, and nobody else can read it.


## 1. Install the Agent Development Kit


In [ ]:
%pip install -q google-adk==2.15.0
import google.adk; print('ADK', google.adk.__version__)


## 2. Get the code

Uses your token to clone the private repository. The token is never displayed.


In [ ]:
import os, pathlib, subprocess, sys
from google.colab import userdata

GITHUB_REPO = 'dipesharma29/agentic-ai-sdlc'
REPO_DIR = pathlib.Path('/content/agentic-ai-sdlc')

try:
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    raise SystemExit('Add a GITHUB_TOKEN secret first - see the checklist above.')

URL = 'https://x-access-token:' + TOKEN + '@github.com/' + GITHUB_REPO + '.git'

def clone(into):
    if into.exists():
        return
    into.parent.mkdir(parents=True, exist_ok=True)
    r = subprocess.run(['git', 'clone', '--quiet', URL, str(into)],
                       capture_output=True, text=True)
    if r.returncode:
        # Redact before showing anything: the URL carries the token.
        raise SystemExit('Clone failed: ' + r.stderr.replace(TOKEN, '***')[:300])

clone(REPO_DIR)
print('cloned', GITHUB_REPO)
print(subprocess.run(['git', '-C', str(REPO_DIR), 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout.strip())


## 3. Sign in to Google Cloud

This uses **your own** Google account. There is no shared key: the agent can reach the
model only because your account is allowed to.


In [ ]:
from google.colab import auth
auth.authenticate_user()

os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'TRUE'
os.environ['GOOGLE_CLOUD_PROJECT'] = 'ai-sdlc-515307'
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'
os.environ['MODEL'] = 'gemini-3.5-flash-lite'
print('signed in; project', os.environ['GOOGLE_CLOUD_PROJECT'])


## 4. Give the agent its own copy to work in

It edits a separate clone and commits to a branch there, so nothing it does can disturb
anything else.


In [ ]:
WORKSPACE = pathlib.Path('/content/workspace/agentic-ai-sdlc')
clone(WORKSPACE)

# The agent's tools read these the moment they are imported, so set them first.
os.environ['TARGET_REPO'] = str(WORKSPACE)
os.environ['BASE_BRANCH'] = 'main'
os.environ['GITHUB_REPO'] = GITHUB_REPO
os.environ['GITHUB_TOKEN'] = TOKEN
print('agent will work in', WORKSPACE)


## 5. Check everything before spending anything

Reaches the model once, confirms the agent's rules file is there, and records the
starting test result - the number the agent has to improve on.


In [ ]:
sys.path.insert(0, str(REPO_DIR / 'poc'))

from google import genai
client = genai.Client(vertexai=True,
                      project=os.environ['GOOGLE_CLOUD_PROJECT'],
                      location=os.environ['GOOGLE_CLOUD_LOCATION'])
reply = client.models.generate_content(model=os.environ['MODEL'],
                                       contents='Reply with one word: ready')
print('model      :', (reply.text or '').strip())
print('rules file :', (WORKSPACE / '.agent.json').exists())

baseline = subprocess.run([sys.executable, '-m', 'unittest', 'discover',
                           '-s', 'poc/sample_site/tests', '-q'],
                          cwd=WORKSPACE, capture_output=True, text=True)
print('tests now  :', baseline.stderr.strip().splitlines()[-1])


## 6. The task

> **The logo image on the home page has no alt text, so screen readers cannot describe
> it. Fix it.**

Notice what that does not say: not which file, not which line, not what to type.

**Watch for** the agent creating a branch, exploring, running the tests to see the
failure, making one change, running them again - then stopping to ask you.


In [ ]:
from google.adk.runners import InMemoryRunner
from google.genai import types
from fix_agent.agent import root_agent

TASK = ('The logo image on the home page has no alt text, so screen readers '
        'cannot describe it. Fix it.')

runner = InMemoryRunner(agent=root_agent, app_name='fix_agent')
session = await runner.session_service.create_session(app_name='fix_agent', user_id='pm')
pending = None

async def send(parts):
    global pending
    async for event in runner.run_async(user_id='pm', session_id=session.id,
                                        new_message=types.Content(role='user', parts=parts)):
        for part in (event.content.parts if event.content else []) or []:
            call = getattr(part, 'function_call', None)
            answer = getattr(part, 'function_response', None)
            if call and call.name == 'adk_request_confirmation':
                pending = call
                original = (call.args or {}).get('originalFunctionCall', {})
                print('\n' + '=' * 62)
                print('WAITING FOR YOUR APPROVAL - nothing has been pushed')
                print('=' * 62)
                print(str(original.get('args', {}))[:900])
            elif call:
                print('  [' + call.name + '] ' + str(dict(call.args or {}))[:120])
            elif answer:
                print('     -> ' + str(answer.response)[:170])
            elif getattr(part, 'text', None):
                print('\n' + part.text)

await send([types.Part(text=TASK)])


## 7. Your decision

The agent has changed files in its own copy and run the tests. It **cannot** push or
open a pull request until you run the next cell.

Run it to approve. To refuse instead, change `True` to `False`: the work is discarded
and nothing leaves this notebook.


In [ ]:
APPROVE = True   # change to False to reject

if not pending:
    print('Nothing is waiting for approval. Check the output above.')
else:
    await send([types.Part(function_response=types.FunctionResponse(
        id=pending.id, name='adk_request_confirmation',
        response={'confirmed': APPROVE}))])


## 8. Check its work yourself

Do not take the agent's summary for it. Look at what actually changed.


In [ ]:
import json, urllib.request

def github(path):
    request = urllib.request.Request(
        path if path.startswith('http') else 'https://api.github.com' + path,
        headers={'Authorization': 'Bearer ' + TOKEN,
                 'Accept': 'application/vnd.github+json'})
    return json.loads(urllib.request.urlopen(request, timeout=30).read())

pulls = github('/repos/' + GITHUB_REPO + '/pulls?state=open&sort=created&direction=desc')
if not pulls:
    print('No open pull request - either you rejected it, or it did not get that far.')
else:
    pr = pulls[0]
    print('PR   :', pr['html_url'])
    print('branch:', pr['head']['ref'], '->', pr['base']['ref'])
    print()
    for changed in github(pr['_links']['self']['href'] + '/files'):
        print(changed['filename'], '(+%d -%d)' % (changed['additions'], changed['deletions']))
        print(changed.get('patch', '')[:900])
        print()


### Three things worth checking

1. **Did the tests go from failing to passing?** Compare step 5 with the agent's output.
2. **Did it change only what it needed to?** On an earlier run it also fixed a second,
   unrelated bug - correct, but more than it was asked for.
3. **Did it leave the tests alone?** It is blocked from editing them, which is what stops
   it "passing" by weakening the check.

---

### What this shows, and what it does not

**Shows:** one agent taking an English sentence through to a reviewed pull request, with
the work checked by the repository's own tests and a human gate before anything leaves
the machine. The agent knows nothing about HTML - a small file in the repo tells it how
to test whatever it is pointed at.

**Does not show:** that it copes with large or unfamiliar codebases, several languages,
or tasks spanning many files. Those are next. `docs/STATE.md` in the repo says what is
proven and what is not, in plain terms.

### If something failed

| Symptom | Cause |
|---|---|
| `Add a GITHUB_TOKEN secret first` | The secret is missing, or *Notebook access* is off |
| `403` / `PERMISSION_DENIED` on the model | Your Google account lacks **Vertex AI User** on `ai-sdlc-515307` |
| Clone failed | Your token lacks access, or you are not a collaborator yet |
| It says the problem is not there | Someone already fixed it. Its reply names the commit it searched |
